# 4a. Demand growth projection, seeds 1 to 8

The baseline demand is the 2011 survey. It is grown at 4.52%/yr, the JICA/DoR 2024 forecast for the Ring Road's main section, with no intervention, over a 6-hour horizon:

* x1.20 (4.2 years), x1.94 (2011 to 2026, about today), x2.32 (2011 to end of 2030).

Assumes peak-hour trips grow one to one with daily traffic, which is not tested. Vehicle positions are logged every 60 steps to fit memory; trip times are unaffected. Three runs at a time, since gridlocked runs need more memory.

In [ ]:
import glob, os, shutil, subprocess, sys, zipfile

def find_project(marker):
    """Copy the project tree that contains `marker` from /kaggle/input to /kaggle/working/sim."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{marker}", recursive=True),
                  key=lambda h: ("build-demand" not in h, len(h)))   # prefer step 1's copy
    if not hits:                                        # dataset delivered as a zip
        for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
            zipfile.ZipFile(z).extractall("/kaggle/working/_unzipped")
        hits = sorted(glob.glob(f"/kaggle/working/_unzipped/**/{marker}", recursive=True), key=len)
    src = hits[0][: -len(marker)].rstrip("/")
    shutil.copytree(src, "/kaggle/working/sim", dirs_exist_ok=True)
    os.chdir("/kaggle/working/sim")
    print("project from", src)

def run(*cmd, tries=1):
    """Run a script, streaming its output; retry resumable drivers if a worker is killed."""
    for attempt in range(tries):
        p = subprocess.Popen([sys.executable, "-u", *cmd], stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end="")
        if p.wait() == 0:
            return
        print(f"exit code {p.returncode}, attempt {attempt + 1} of {tries}")
    raise RuntimeError(" ".join(cmd))

find_project("data/jica_net/od_matrix.csv")
if os.path.exists("results/jica_projection.partial.csv"): os.remove("results/jica_projection.partial.csv")

In [ ]:
!pip install -q uxsim==1.14.2

In [ ]:
run("scripts/project_jica.py", "--first-seed", "1", "--seeds", "8", "--jobs", "3", tries=3)
shutil.copy("results/jica_projection.partial.csv", "/kaggle/working/projection_seeds1-8.csv")